# 4. Dataset Splitting and Training Preparation

This notebook prepares the non-normalized ECG signals and creates the final patient-level dataset split.

Three cross-validation strategies are compared within the development set: `GroupKFold`, `StratifiedGroupKFold`, and patient-level `StratifiedKFold`.
The current saving cells persist the `StratifiedGroupKFold` folds; the other strategies are kept for comparison.

## 4.1 Loading the preprocessed data

This section loads the non-normalized signals and metadata stored in the `.npz` file.

The signals are stored in `X`, the original labels in `y_original`, and the patient and lead identifiers in `patient_ids` and `leads`.

## Contents

1. Load the preprocessed dataset
2. Validate signal–metadata alignment
3. Convert labels and define patient groups
4. Create the patient-level development/test split
5. Compare cross-validation strategies within development
6. Save and verify the split files
7. Variable reference

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split


nonormalized_data_path = (
    Path("data")
    / "nonormalized_signals_clean.npz"
)

if not nonormalized_data_path.is_file():
    raise FileNotFoundError(
        f"File not found: {nonormalized_data_path}"
    )


nonormalized_data = np.load(
    nonormalized_data_path
)

required_keys = {
    "signals",
    "label",
    "patient_id",
    "lead",
    "mat_path",
}

missing_keys = required_keys - set(
    nonormalized_data.files
)

if missing_keys:
    raise KeyError(
        f"Missing keys in the .npz file: "
        f"{sorted(missing_keys)}"
    )

print(
    "Available keys:",
    nonormalized_data.files,
)

# Extract variables before creating dataset_index.
X = np.asarray(
    nonormalized_data["signals"]
)

y_original = np.asarray(
    nonormalized_data["label"],
    dtype=int,
)

patient_ids = np.asarray(
    nonormalized_data["patient_id"],
    dtype=str,
)

leads = np.asarray(
    nonormalized_data["lead"],
    dtype=str,
)

mat_paths = np.asarray(
    nonormalized_data["mat_path"],
    dtype=str,
)

# Reconstruct the metadata index.
dataset_index = pd.DataFrame(
    {
        "patient_id": patient_ids,
        "lead": leads,
        "label": y_original,
        "mat_path": mat_paths,
    }
).reset_index(
    drop=True
)

ModuleNotFoundError: No module named 'numpy'

## 4.2 Checking signal–metadata alignment

This section verifies that all arrays contain the same number of records. Row `i` of `X` must correspond to row `i` of `dataset_index`.

The signals and metadata are not reordered.

In [2]:
assert X.ndim == 2
assert X.shape[1] == 252
assert len(X) == len(dataset_index)
assert len(y_original) == len(X)
assert len(patient_ids) == len(X)
assert len(leads) == len(X)
assert len(mat_paths) == len(X)


print(
    "La longitud de X y de dataset_index coincide."
)

print(
    f"Total signals: {len(X):,}"
)

print(
    f"Patients: "
    f"{dataset_index['patient_id'].nunique():,}"
)

# Create a small table to inspect the correspondence.
first_rows = dataset_index.iloc[:5].copy()

first_rows.insert(
    0,
    "index",
    first_rows.index,
)

columns_to_display = [
    "index",
    "patient_id",
    "lead",
    "label",
]

if "mat_path" in first_rows.columns:
    columns_to_display.append("mat_path")

print(
    "First signal and metadata rows:"
)

display(
    first_rows[columns_to_display]
)

# Compute general dataset statistics.
number_of_positive_signals = (
    y_original == 1
).sum()

number_of_negative_signals = (
    y_original == 99
).sum()

print(
    f"Positive signals: "
    f"{number_of_positive_signals:,}"
)

print(
    f"Negative signals: "
    f"{number_of_negative_signals:,}"
)


print("Distribution by lead:")
display(
    pd.Series(leads)
    .value_counts()
    .sort_index()
)



La longitud de X y de dataset_index coincide.
Total signals: 10,577
Patients: 33
First signal and metadata rows:


,index,patient_id,lead,label,mat_path
0,0,BH0002,V1,99,/Users/ainhoa.ns/Desktop/TurraFM/notebooks/dat...
1,1,BH0002,V2,99,/Users/ainhoa.ns/Desktop/TurraFM/notebooks/dat...
2,2,BH0002,V3,99,/Users/ainhoa.ns/Desktop/TurraFM/notebooks/dat...
3,3,BH0002,V4,99,/Users/ainhoa.ns/Desktop/TurraFM/notebooks/dat...
4,4,BH0002,V5,1,/Users/ainhoa.ns/Desktop/TurraFM/notebooks/dat...


Positive signals: 2,007
Negative signals: 8,570
Distribution by lead:


V1    1760
V2    1773
V3    1761
V4    1755
V5    1769
V6    1759
Name: count, dtype: int64

## 4.3 Preparing labels and patient groups

The original labels are converted to a binary representation:

- `1` remains `1`: Brugada.
- `99` becomes `0`: non-Brugada.

The original labels are preserved in `y_original`.

The `groups` array is also created to associate each signal with its patient. These groups prevent signals from the same patient from being assigned to different subsets.

In [3]:
# Convert the original labels to binary labels.
expected_labels = {1, 99}
found_labels = set(np.unique(y_original))

unexpected_labels = found_labels - expected_labels

if unexpected_labels:
    raise ValueError(
        f"Unexpected labels found: "
        f"{sorted(unexpected_labels)}"
    )

label_mapping = {
    1: 1,
    99: 0,
}

Y = np.array(
    [
        label_mapping[int(label)]
        for label in y_original
    ],
    dtype=int,
)

# Groups are the patient identifiers.
groups = patient_ids.copy()

# Check the binary labels.
print(
    f"Unique values in Y: "
    f"{np.unique(Y)}"
)

print(
    f"Positive signals: "
    f"{(Y == 1).sum():,}"
)

print(
    f"Negative signals: "
    f"{(Y == 0).sum():,}"
)

print(
    f"Positive proportion: "
    f"{np.mean(Y == 1):.4f}"
)

print(
    f"Negative proportion: "
    f"{np.mean(Y == 0):.4f}"
)

# Check that there is one group per signal.
assert len(groups) == len(X)

print(
    f"Unique patients in groups: "
    f"{len(np.unique(groups))}"
)

# The split must be performed by patient rather than by signal.
# Signals from the same patient may be correlated.
# Splitting them individually could cause information leakage.

Unique values in Y: [0 1]
Positive signals: 2,007
Negative signals: 8,570
Positive proportion: 0.1898
Negative proportion: 0.8102
Unique patients in groups: 33


## 4.4 Creating an independent patient-level test set

Approximately 20% of the patients are reserved for the test set.

The split is performed using `patient_id`, rather than randomly splitting individual signals. Therefore, no patient can appear simultaneously in development and test.
This random split does not guarantee balanced class proportions between development and test.

In [4]:
# Obtain the list of unique patients.
unique_patient_ids = np.unique(
    patient_ids
)

development_ids, test_ids = (
    train_test_split(
        unique_patient_ids,
        test_size=0.20,
        random_state=43,
        shuffle=True,
    )
)


print(
    f"Total patients: "
    f"{len(unique_patient_ids)}"
)

print(
    f"Development patients: "
    f"{len(development_ids)}"
)

print(
    f"Test patients: "
    f"{len(test_ids)}"
)

print(
    "Patients included in test:"
)

print(
    sorted(test_ids)
)

Total patients: 33
Development patients: 26
Test patients: 7
Patients included in test:
['BH0008', 'BH0009', 'BH0010', 'BH0012', 'BH0017', 'BH0026', 'BH0029']


In [5]:
# Create masks assigning every signal to the test or development subset.

test_mask = np.isin(
    patient_ids,
    test_ids,
)

development_mask = np.isin(
    patient_ids,
    development_ids,
)

assert not (
    set(development_ids)
    & set(test_ids)
)

assert (
    development_mask.sum()
    + test_mask.sum()
    == len(X)
)

print(
    "✓ No patients are shared between development and test."
)


✓ No patients are shared between development and test.


In [6]:
# Create variables for the development and test subsets.

groups = patient_ids

development_indices = np.where(
    development_mask
)[0]

test_indices = np.where(
    test_mask
)[0]

# Development variables.
X_dev = X[
    development_indices
]

y_dev = Y[
    development_indices
]

groups_dev = groups[
    development_indices
]

leads_dev = leads[
    development_indices
]

# Test variables.
X_test = X[
    test_indices
]

y_test = Y[
    test_indices
]

groups_test = groups[
    test_indices
]

leads_test = leads[
    test_indices
]

In [7]:
# Build a comparison summary for development and test.
split_summary = pd.DataFrame(
    [
        {
            "subset": "development",
            "signals": len(X_dev),
            "patients": len(np.unique(groups_dev)),
            "positive_signals": (y_dev == 1).sum(),
            "negative_signals": (y_dev == 0).sum(),
            "positive_percentage": (
                100 * np.mean(y_dev == 1)
            ),
            "negative_percentage": (
                100 * np.mean(y_dev == 0)
            ),
        },
        {
            "subset": "test",
            "signals": len(X_test),
            "patients": len(np.unique(groups_test)),
            "positive_signals": (y_test == 1).sum(),
            "negative_signals": (y_test == 0).sum(),
            "positive_percentage": (
                100 * np.mean(y_test == 1)
            ),
            "negative_percentage": (
                100 * np.mean(y_test == 0)
            ),
        },
    ]
)

split_summary[
    [
        "positive_percentage",
        "negative_percentage",
    ]
] = split_summary[
    [
        "positive_percentage",
        "negative_percentage",
    ]
].round(2)

display(split_summary)


print(
    "Patients included in test:"
)

print(
    sorted(np.unique(groups_test))
)

,subset,signals,patients,positive_signals,negative_signals,positive_percentage,negative_percentage
0,development,9359,26,1740,7619,18.59,81.41
1,test,1218,7,267,951,21.92,78.08


Patients included in test:
['BH0008', 'BH0009', 'BH0010', 'BH0012', 'BH0017', 'BH0026', 'BH0029']


### Initial split summary

This initial split is performed at patient level to prevent information leakage between development and test.

The resulting development and test subsets are summarized below.
The selected cross-validation folds are saved later in the notebook.

In [8]:
assert len(
    development_indices
) + len(
    test_indices
) == len(X)

assert set(
    development_indices
).isdisjoint(
    set(test_indices)
)

assert set(
    development_indices
).union(
    set(test_indices)
) == set(range(len(X)))


print("Dataset total:")
print(
    f"- Signals: {len(X):,}"
)

print(
    f"- Patients: {len(np.unique(groups)):,}"
)

print(
    f"- Positive signals: {(Y == 1).sum():,}"
)

print(
    f"- Negative signals: {(Y == 0).sum():,}"
)

print("\nDevelopment:")
print(
    f"- Signals: {len(X_dev):,}"
)

print(
    f"- Patients: {len(np.unique(groups_dev)):,}"
)

print(
    f"- Positive signals: {(y_dev == 1).sum():,}"
)

print(
    f"- Negative signals: {(y_dev == 0).sum():,}"
)
# Percentage of positive signals in development.
dev_positive_percentage = (
    (y_dev == 1).sum()
    / len(y_dev)
    * 100
)

print(
    f"Positive percentage in development: "
    f"{dev_positive_percentage:.2f}%"
)

print("\nTest:")
print(
    f"- Signals: {len(X_test):,}"
)

print(
    f"- Patients: {len(np.unique(groups_test)):,}"
)

print(
    f"- Positive signals: {(y_test == 1).sum():,}"
)

print(
    f"- Negative signals: {(y_test == 0).sum():,}"
)

print(
    "- Included patients:"
)

print(
    sorted(np.unique(groups_test))
)

# Percentage of positive signals in test.
test_positive_percentage = (
    (y_test == 1).sum()
    / len(y_test)
    * 100
)


print(
    f"Positive percentage in test: "
    f"{test_positive_percentage:.2f}%"
)



Dataset total:
- Signals: 10,577
- Patients: 33
- Positive signals: 2,007
- Negative signals: 8,570

Development:
- Signals: 9,359
- Patients: 26
- Positive signals: 1,740
- Negative signals: 7,619
Positive percentage in development: 18.59%

Test:
- Signals: 1,218
- Patients: 7
- Positive signals: 267
- Negative signals: 951
- Included patients:
['BH0008', 'BH0009', 'BH0010', 'BH0012', 'BH0017', 'BH0026', 'BH0029']
Positive percentage in test: 21.92%


## 4.5 Cross-validation within the development set
The following sections compare three strategies within the development set.

`GroupKFold`, `StratifiedGroupKFold`, and `StratifiedKFold` applied at patient level.

Each strategy is analyzed separately. The notebook reports their properties before saving the selected fold representation.

### 4.5.1. Exploring the development set

This section analyzes the development set exclusively.

The test set remains locked and is not used to compute statistics or choose a cross-validation strategy.

#### 4.5.1.1 Signal distribution by patient

A one-row-per-patient table is created using `groups_dev` and `y_dev`.

The `has_positive` column indicates whether a patient contributes at least one positive signal.

In [9]:
# Create a temporary table with patient identifiers and labels.
development_labels = pd.DataFrame(
    {
        "patient_id": groups_dev,
        "label": y_dev,
    }
)

# Compute total and positive signals per patient.
development_distribution = (
    development_labels
    .groupby("patient_id")
    .agg(
        total_signals=("label", "size"),
        positive_signals=("label", "sum"),
    )
)

# Negative signals are those with label 0.
development_distribution[
    "negative_signals"
] = (
    development_distribution[
        "total_signals"
    ]
    - development_distribution[
        "positive_signals"
    ]
)

# Compute the positive-signal percentage for each patient.
development_distribution[
    "positive_percentage"
] = (
    100
    * development_distribution[
        "positive_signals"
    ]
    / development_distribution[
        "total_signals"
    ]
)

# Indicate whether the patient contributes at least one positive signal.
development_distribution[
    "has_positive"
] = (
    development_distribution[
        "positive_signals"
    ]
    > 0
).astype(int)

# Reorder the columns and patient index.
development_distribution = (
    development_distribution[
        [
            "positive_signals",
            "negative_signals",
            "total_signals",
            "positive_percentage",
            "has_positive",
        ]
    ]
    .sort_index()
)

display(
    development_distribution
)

# Total number of development patients.
development_patient_count = len(
    development_distribution
)

# Patients contributing at least one positive signal.
development_positive_patient_count = (
    development_distribution[
        "has_positive"
    ]
    == 1
).sum()

# Patients with no positive signals.
development_negative_only_patient_count = (
    development_distribution[
        "has_positive"
    ]
    == 0
).sum()

print(
    f"Total development patients: "
    f"{development_patient_count}"
)

print(
    f"Patients with at least one positive signal: "
    f"{development_positive_patient_count}"
)

print(
    f"Patients with no positive signals: "
    f"{development_negative_only_patient_count}"
)

,positive_signals,negative_signals,total_signals,positive_percentage,has_positive
patient_id,,,,,
BH0002,364,1274,1638,22.222222,1
BH0003,0,612,612,0.000000,0
BH0004,126,252,378,33.333333,1
BH0005,246,0,246,100.000000,1
BH0006,0,252,252,0.000000,0
BH0007,57,123,180,31.666667,1
BH0014,0,252,252,0.000000,0
BH0015,0,318,318,0.000000,0
BH0016,0,72,72,0.000000,0


Total development patients: 26
Patients with at least one positive signal: 12
Patients with no positive signals: 14


#### 4.5.1.2 Class heterogeneity across patients

Patients are divided into three categories:

1. Patients with no positive signals.
2. Patients with both positive and negative signals.
3. Patients whose signals are all positive.

This analysis reveals whether the classes are concentrated in particular patients.

In [10]:
# Patients whose signals are all negative.
patients_with_only_negative_signals = (
    development_distribution[
        development_distribution[
            "positive_signals"
        ]
        == 0
    ]
)

# Patients with both positive and negative signals.
patients_with_both_classes = (
    development_distribution[
        (
            development_distribution[
                "positive_signals"
            ]
            > 0
        )
        & (
            development_distribution[
                "negative_signals"
            ]
            > 0
        )
    ]
)

# Patients whose signals are all positive.
patients_with_only_positive_signals = (
    development_distribution[
        (
            development_distribution[
                "positive_signals"
            ]
            > 0
        )
        & (
            development_distribution[
                "negative_signals"
            ]
            == 0
        )
    ]
)

print(
    "Patients with no positive signals:"
)

display(
    patients_with_only_negative_signals
)

print(
    "Patients with positive and negative signals:"
)

display(
    patients_with_both_classes
)

print(
    "Patients with only positive signals:"
)

display(
    patients_with_only_positive_signals
)

print(
    f"Patients with no positive signals: "
    f"{len(patients_with_only_negative_signals)}"
)

print(
    f"Patients with both classes: "
    f"{len(patients_with_both_classes)}"
)

print(
    f"Patients with only positive signals: "
    f"{len(patients_with_only_positive_signals)}"
)

Patients with no positive signals:


,positive_signals,negative_signals,total_signals,positive_percentage,has_positive
patient_id,,,,,
BH0003,0,612,612,0.0,0
BH0006,0,252,252,0.0,0
BH0014,0,252,252,0.0,0
BH0015,0,318,318,0.0,0
BH0016,0,72,72,0.0,0
BH0018,0,126,126,0.0,0
BH0020,0,300,300,0.0,0
BH0028,0,468,468,0.0,0
BH0031,0,384,384,0.0,0


Patients with positive and negative signals:


,positive_signals,negative_signals,total_signals,positive_percentage,has_positive
patient_id,,,,,
BH0002,364,1274,1638,22.222222,1
BH0004,126,252,378,33.333333,1
BH0007,57,123,180,31.666667,1
BH0019,118,386,504,23.412698,1
BH0022,64,236,300,21.333333,1
BH0023,78,48,126,61.904762,1
BH0025,5,373,378,1.322751,1
BH0027,2,454,456,0.438596,1
BH0030,118,386,504,23.412698,1


Patients with only positive signals:


,positive_signals,negative_signals,total_signals,positive_percentage,has_positive
patient_id,,,,,
BH0005,246,0,246,100.0,1


Patients with no positive signals: 14
Patients with both classes: 11
Patients with only positive signals: 1


#### 4.5.1.3 Lead distribution within development

The number and percentage of signals are computed for each lead in the development set.

This descriptive analysis does not modify the split.

In [11]:
# Create a table with the lead and label of each signal.
development_lead_labels = pd.DataFrame(
    {
        "lead": leads_dev,
        "label": y_dev,
    }
)

# Count positive and negative signals by lead.
class_by_lead = pd.crosstab(
    development_lead_labels["lead"],
    development_lead_labels["label"],
)

# Aseguramos que aparezcan las seis derivaciones
# and both classes even if one has zero signals.
class_by_lead = class_by_lead.reindex(
    index=[
        "V1",
        "V2",
        "V3",
        "V4",
        "V5",
        "V6",
    ],
    columns=[0, 1],
    fill_value=0,
)

# Rename columns according to the label meaning.
class_by_lead.columns = [
    "negative_signals",
    "positive_signals",
]

# Compute the total number of signals per lead.
class_by_lead["total_signals"] = (
    class_by_lead["negative_signals"]
    + class_by_lead["positive_signals"]
)

# Compute the positive-signal percentage for each lead.
class_by_lead["positive_percentage"] = (
    100
    * class_by_lead["positive_signals"]
    / class_by_lead["total_signals"]
)

class_by_lead[
    "positive_percentage"
] = class_by_lead[
    "positive_percentage"
].round(2)

# Convert the index into a column named lead.
class_by_lead = (
    class_by_lead
    .reset_index()
)

display(
    class_by_lead
)

,lead,negative_signals,positive_signals,total_signals,positive_percentage
0,V1,1166,391,1557,25.11
1,V2,1300,270,1570,17.20
2,V3,1418,140,1558,8.99
3,V4,1271,281,1552,18.11
4,V5,1284,282,1566,18.01
5,V6,1180,376,1556,24.16


### 4.5.2 GroupKFold
This section creates five patient-grouped cross-validation folds within the development set.

#### 4.5.2.1 Creating the folds

`GroupKFold` is used to split the development set exclusively.

The `groups_dev` parameter ensures that all signals from the same patient remain in the same fold.

In [12]:
from sklearn.model_selection import GroupKFold


# Create GroupKFold with five splits.
group_kfold = GroupKFold(
    n_splits=5
)

print(
    "GroupKFold creado con "
    f"{group_kfold.n_splits} folds."
)

GroupKFold creado con 5 folds.


#### 4.5.2.2. Creating and storing the five fold indices

The five folds are generated using the development set only.

The indices are relative to `X_dev`, `y_dev`, `groups_dev`, and `leads_dev`. The test set is not involved.

In [13]:
# List where the indices of each fold will be stored.
groupk_folds = []

for fold_number, (
    train_indices,
    val_indices,
) in enumerate(
    group_kfold.split(
        X_dev,
        y_dev,
        groups=groups_dev,
    ),
    start=1,
):
    groupk_folds.append(
        {
            "fold": fold_number,
            "train_indices": train_indices,
            "val_indices": val_indices,
        }
    )

print(
    f"Folds generated: {len(groupk_folds)}"
)

Folds generated: 5


#### 4.5.2.3. Checking for patient leakage

This section verifies that no patient appears simultaneously in training and validation within the same fold.

In [14]:
for fold_data in groupk_folds:
    fold_number = fold_data["fold"]
    train_indices = fold_data["train_indices"]
    val_indices = fold_data["val_indices"]

    # Patients present in each subset.
    train_patients = np.unique(
        groups_dev[train_indices]
    )

    val_patients = np.unique(
        groups_dev[val_indices]
    )

    # Patients shared between training and validation.
    shared_patients = set(
        train_patients
    ).intersection(
        set(val_patients)
    )

    # Check that no patients are shared.
    assert set(
        train_patients
    ).isdisjoint(
        set(val_patients)
    )

    # Check that all indices are included.
    assert (
        len(train_indices)
        + len(val_indices)
        == len(X_dev)
    )

    # Check that no indices are repeated.
    assert set(
        train_indices
    ).isdisjoint(
        set(val_indices)
    )

    print(f"Fold {fold_number}:")
    print(
        f"- Training patients: "
        f"{len(train_patients)}"
    )
    print(
        f"- Validation patients: "
        f"{len(val_patients)}"
    )
    print(
        f"- Shared patients: "
        f"{len(shared_patients)}"
    )
    print()

Fold 1:
- Training patients: 23
- Validation patients: 3
- Shared patients: 0

Fold 2:
- Training patients: 20
- Validation patients: 6
- Shared patients: 0

Fold 3:
- Training patients: 21
- Validation patients: 5
- Shared patients: 0

Fold 4:
- Training patients: 20
- Validation patients: 6
- Shared patients: 0

Fold 5:
- Training patients: 20
- Validation patients: 6
- Shared patients: 0



#### 4.5.2.4. Signal and class summary by fold

Training and validation statistics are computed for each fold.

The folds are not modified if their class proportions differ.

In [15]:
fold_summary_records = []

for fold_data in groupk_folds:
    fold_number = fold_data["fold"]
    train_indices = fold_data["train_indices"]
    val_indices = fold_data["val_indices"]

    # Extract labels for each subset.
    train_labels = y_dev[train_indices]
    val_labels = y_dev[val_indices]

    # Compute training-signal statistics.
    train_signals = len(train_indices)
    train_positive = (
        train_labels == 1
    ).sum()

    train_negative = (
        train_labels == 0
    ).sum()

    train_positive_percentage = (
        100
        * train_positive
        / train_signals
    )

    # Compute validation-signal statistics.
    val_signals = len(val_indices)
    val_positive = (
        val_labels == 1
    ).sum()

    val_negative = (
        val_labels == 0
    ).sum()

    val_positive_percentage = (
        100
        * val_positive
        / val_signals
    )

    # Compute the number of patients in each subset.
    train_patients = np.unique(
        groups_dev[train_indices]
    )

    val_patients = np.unique(
        groups_dev[val_indices]
    )

    fold_summary_records.append(
        {
            "fold": fold_number,
            "train_signals": train_signals,
            "val_signals": val_signals,
            "train_patients": len(
                train_patients
            ),
            "val_patients": len(
                val_patients
            ),
            "train_positive": train_positive,
            "train_negative": train_negative,
            "val_positive": val_positive,
            "val_negative": val_negative,
            "train_positive_percentage": (
                train_positive_percentage
            ),
            "val_positive_percentage": (
                val_positive_percentage
            ),
        }
    )

fold_summary_df = pd.DataFrame(
    fold_summary_records
)

fold_summary_df[
    [
        "train_positive_percentage",
        "val_positive_percentage",
    ]
] = fold_summary_df[
    [
        "train_positive_percentage",
        "val_positive_percentage",
    ]
].round(2)

display(
    fold_summary_df
)

,fold,train_signals,val_signals,train_patients,val_patients,train_positive,train_negative,val_positive,val_negative,train_positive_percentage,val_positive_percentage
0,1,7474,1885,23,3,1319,6155,421,1464,17.65,22.33
1,2,7480,1879,20,6,1240,6240,500,1379,16.58,26.61
2,3,7497,1862,21,5,1494,6003,246,1616,19.93,13.21
3,4,7505,1854,20,6,1434,6071,306,1548,19.11,16.50
4,5,7480,1879,20,6,1473,6007,267,1612,19.69,14.21


#### 4.5.2.5. Patients contributing positive signals

The number of training and validation patients contributing at least one positive signal is computed for each fold.

In [16]:
train_positive_patient_counts = []
val_positive_patient_counts = []

for fold_data in groupk_folds:
    train_indices = fold_data["train_indices"]
    val_indices = fold_data["val_indices"]

    train_labels = y_dev[train_indices]
    val_labels = y_dev[val_indices]

    train_groups = groups_dev[train_indices]
    val_groups = groups_dev[val_indices]

    # Training patients contributing at least one positive signal.
    train_positive_patients = np.unique(
        train_groups[
            train_labels == 1
        ]
    )

    # Validation patients contributing at least one positive signal.
    val_positive_patients = np.unique(
        val_groups[
            val_labels == 1
        ]
    )

    train_positive_patient_counts.append(
        len(train_positive_patients)
    )

    val_positive_patient_counts.append(
        len(val_positive_patients)
    )

# Add the results to the fold summary.
fold_summary_df[
    "train_positive_patients"
] = train_positive_patient_counts

fold_summary_df[
    "val_positive_patients"
] = val_positive_patient_counts

display(
    fold_summary_df[
        [
            "fold",
            "train_positive_patients",
            "val_positive_patients",
        ]
    ]
)

,fold,train_positive_patients,val_positive_patients
0,1,10,2
1,2,11,1
2,3,11,1
3,4,9,3
4,5,7,5


#### 4.5.2.6. Checking the presence of both classes

The labels present in training and validation are checked for each fold.

A warning is displayed if a subset does not contain both classes. No fold is modified.

In [17]:
class_presence_records = []

train_classes_all_present = True
val_classes_all_present = True

for fold_data in groupk_folds:
    fold_number = fold_data["fold"]
    train_indices = fold_data["train_indices"]
    val_indices = fold_data["val_indices"]

    train_classes = np.unique(
        y_dev[train_indices]
    )

    val_classes = np.unique(
        y_dev[val_indices]
    )

    train_has_both_classes = (
        set(train_classes) == {0, 1}
    )

    val_has_both_classes = (
        set(val_classes) == {0, 1}
    )

    if not train_has_both_classes:
        train_classes_all_present = False

    if not val_has_both_classes:
        val_classes_all_present = False

    class_presence_records.append(
        {
            "fold": fold_number,
            "train_classes": train_classes,
            "val_classes": val_classes,
            "train_has_both_classes": (
                train_has_both_classes
            ),
            "val_has_both_classes": (
                val_has_both_classes
            ),
        }
    )

class_presence_df = pd.DataFrame(
    class_presence_records
)

display(
    class_presence_df
)

if not train_classes_all_present:
    print(
        "Warning: a training fold does not contain both classes."
    )

if not val_classes_all_present:
    print(
        "Warning: a validation fold does not contain both classes."
    )

if (
    train_classes_all_present
    and val_classes_all_present
):
    print(
        "All folds contain both classes "
        "in training and validation."
    )

,fold,train_classes,val_classes,train_has_both_classes,val_has_both_classes
0,1,"[0, 1]","[0, 1]",True,True
1,2,"[0, 1]","[0, 1]",True,True
2,3,"[0, 1]","[0, 1]",True,True
3,4,"[0, 1]","[0, 1]",True,True
4,5,"[0, 1]","[0, 1]",True,True


All folds contain both classes in training and validation.


With `GroupKFold`, the split is correct with respect to patient independence:

- No patients are shared between training and validation in any fold.
- Both subsets contain both classes in every fold.
- The number of signals is relatively stable across folds.

However, the distribution of positive signals is not well balanced.

This is preferable to randomly splitting signals because it prevents patient leakage, but it is not ideal for model evaluation because of class imbalance.

### 4.5.3 StratifiedGroupKFold

`StratifiedGroupKFold` is used to create five folds within the development set.

This strategy attempts to preserve class proportions while keeping all signals from the same patient together. The test set remains locked.

In [18]:
from sklearn.model_selection import StratifiedGroupKFold


# Create the strategy with five folds.
stratified_group_kfold = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=43,
)

print(
    "StratifiedGroupKFold creado con "
    f"{stratified_group_kfold.n_splits} folds."
)

StratifiedGroupKFold creado con 5 folds.


#### 4.5.3.1 Generating and storing the fold indices

The five folds are generated using only `X_dev`, `y_dev`, and `groups_dev`.

The indices are relative to the development set, and no file is saved at this stage.

In [19]:
# Store the StratifiedGroupKFold indices.
# in a list independent from the previous folds.
stratified_cv_folds = []

for fold_number, (
    stratified_train_indices,
    stratified_val_indices,
) in enumerate(
    stratified_group_kfold.split(
        X_dev,
        y_dev,
        groups=groups_dev,
    ),
    start=1,
):
    stratified_cv_folds.append(
        {
            "fold": fold_number,
            "train_indices": stratified_train_indices,
            "val_indices": stratified_val_indices,
        }
    )

print(
    f"Stratified folds generated: "
    f"{len(stratified_cv_folds)}"
)

Stratified folds generated: 5


#### 4.5.3.2 Checking for patient leakage

This section verifies that no patient appears simultaneously in training and validation within the same fold.

It also verifies that all development indices are included and that no indices are repeated.

In [20]:
for stratified_fold_data in stratified_cv_folds:
    stratified_fold_number = (
        stratified_fold_data["fold"]
    )

    stratified_train_indices = (
        stratified_fold_data["train_indices"]
    )

    stratified_val_indices = (
        stratified_fold_data["val_indices"]
    )

    # Obtain the patients in each subset.
    stratified_train_patients = np.unique(
        groups_dev[
            stratified_train_indices
        ]
    )

    stratified_val_patients = np.unique(
        groups_dev[
            stratified_val_indices
        ]
    )

    # Check that no patients are shared.
    stratified_shared_patients = set(
        stratified_train_patients
    ).intersection(
        set(stratified_val_patients)
    )

    assert set(
        stratified_train_patients
    ).isdisjoint(
        set(stratified_val_patients)
    )

    # Check that all signals are used.
    assert (
        len(stratified_train_indices)
        + len(stratified_val_indices)
        == len(X_dev)
    )

    # Check that no indices are repeated.
    assert set(
        stratified_train_indices
    ).isdisjoint(
        set(stratified_val_indices)
    )

    print(
        f"Fold estratificado "
        f"{stratified_fold_number}:"
    )

    print(
        f"- Training patients: "
        f"{len(stratified_train_patients)}"
    )

    print(
        f"- Validation patients: "
        f"{len(stratified_val_patients)}"
    )

    print(
        f"- Shared patients: "
        f"{len(stratified_shared_patients)}"
    )

    print()

# Compute training and validation statistics.
stratified_fold_summary_records = []

for stratified_fold_data in stratified_cv_folds:
    stratified_fold_number = (
        stratified_fold_data["fold"]
    )

    stratified_train_indices = (
        stratified_fold_data["train_indices"]
    )

    stratified_val_indices = (
        stratified_fold_data["val_indices"]
    )

    stratified_train_labels = y_dev[
        stratified_train_indices
    ]

    stratified_val_labels = y_dev[
        stratified_val_indices
    ]

    # Training-subset statistics.
    stratified_train_signals = len(
        stratified_train_indices
    )

    stratified_train_patients = len(
        np.unique(
            groups_dev[
                stratified_train_indices
            ]
        )
    )

    stratified_train_positive = (
        stratified_train_labels == 1
    ).sum()

    stratified_train_negative = (
        stratified_train_labels == 0
    ).sum()

    stratified_train_positive_percentage = (
        100
        * stratified_train_positive
        / stratified_train_signals
    )

    # Validation-subset statistics.
    stratified_val_signals = len(
        stratified_val_indices
    )

    stratified_val_patients = len(
        np.unique(
            groups_dev[
                stratified_val_indices
            ]
        )
    )

    stratified_val_positive = (
        stratified_val_labels == 1
    ).sum()

    stratified_val_negative = (
        stratified_val_labels == 0
    ).sum()

    stratified_val_positive_percentage = (
        100
        * stratified_val_positive
        / stratified_val_signals
    )

    stratified_fold_summary_records.append(
        {
            "fold": stratified_fold_number,
            "train_signals": stratified_train_signals,
            "val_signals": stratified_val_signals,
            "train_patients": stratified_train_patients,
            "val_patients": stratified_val_patients,
            "train_positive": stratified_train_positive,
            "train_negative": stratified_train_negative,
            "val_positive": stratified_val_positive,
            "val_negative": stratified_val_negative,
            "train_positive_percentage": (
                stratified_train_positive_percentage
            ),
            "val_positive_percentage": (
                stratified_val_positive_percentage
            ),
        }
    )

stratified_fold_summary_df = pd.DataFrame(
    stratified_fold_summary_records
)

stratified_fold_summary_df[
    [
        "train_positive_percentage",
        "val_positive_percentage",
    ]
] = stratified_fold_summary_df[
    [
        "train_positive_percentage",
        "val_positive_percentage",
    ]
].round(2)

display(
    stratified_fold_summary_df
)

stratified_train_positive_patient_counts = []
stratified_val_positive_patient_counts = []

for stratified_fold_data in stratified_cv_folds:
    stratified_train_indices = (
        stratified_fold_data["train_indices"]
    )

    stratified_val_indices = (
        stratified_fold_data["val_indices"]
    )

    stratified_train_labels = y_dev[
        stratified_train_indices
    ]

    stratified_val_labels = y_dev[
        stratified_val_indices
    ]

    stratified_train_groups = groups_dev[
        stratified_train_indices
    ]

    stratified_val_groups = groups_dev[
        stratified_val_indices
    ]

    # Training patients with at least one positive signal.
    stratified_train_positive_patients = np.unique(
        stratified_train_groups[
            stratified_train_labels == 1
        ]
    )

    # Validation patients with at least one positive signal.
    stratified_val_positive_patients = np.unique(
        stratified_val_groups[
            stratified_val_labels == 1
        ]
    )

    stratified_train_positive_patient_counts.append(
        len(stratified_train_positive_patients)
    )

    stratified_val_positive_patient_counts.append(
        len(stratified_val_positive_patients)
    )

stratified_fold_summary_df[
    "train_positive_patients"
] = stratified_train_positive_patient_counts

stratified_fold_summary_df[
    "val_positive_patients"
] = stratified_val_positive_patient_counts

display(
    stratified_fold_summary_df[
        [
            "fold",
            "train_positive_patients",
            "val_positive_patients",
        ]
    ]
)

stratified_class_presence_records = []

stratified_train_classes_all_present = True
stratified_val_classes_all_present = True

for stratified_fold_data in stratified_cv_folds:
    stratified_fold_number = (
        stratified_fold_data["fold"]
    )

    stratified_train_indices = (
        stratified_fold_data["train_indices"]
    )

    stratified_val_indices = (
        stratified_fold_data["val_indices"]
    )

    stratified_train_classes = np.unique(
        y_dev[
            stratified_train_indices
        ]
    )

    stratified_val_classes = np.unique(
        y_dev[
            stratified_val_indices
        ]
    )

    stratified_train_has_both_classes = (
        set(stratified_train_classes)
        == {0, 1}
    )

    stratified_val_has_both_classes = (
        set(stratified_val_classes)
        == {0, 1}
    )

    if not stratified_train_has_both_classes:
        stratified_train_classes_all_present = False

    if not stratified_val_has_both_classes:
        stratified_val_classes_all_present = False

    stratified_class_presence_records.append(
        {
            "fold": stratified_fold_number,
            "train_classes": stratified_train_classes,
            "val_classes": stratified_val_classes,
            "train_has_both_classes": (
                stratified_train_has_both_classes
            ),
            "val_has_both_classes": (
                stratified_val_has_both_classes
            ),
        }
    )

stratified_class_presence_df = pd.DataFrame(
    stratified_class_presence_records
)

display(
    stratified_class_presence_df
)

if not stratified_train_classes_all_present:
    print(
        "Warning: a stratified training fold does not contain both classes."
    )

if not stratified_val_classes_all_present:
    print(
        "Warning: a stratified validation fold does not contain both classes."
    )

stratified_validation_patients_by_fold = {}

for stratified_fold_data in stratified_cv_folds:
    stratified_fold_number = (
        stratified_fold_data["fold"]
    )

    stratified_val_indices = (
        stratified_fold_data["val_indices"]
    )

    stratified_val_patients = np.unique(
        groups_dev[
            stratified_val_indices
        ]
    )

    stratified_validation_patients_by_fold[
        stratified_fold_number
    ] = sorted(
        stratified_val_patients
    )

    print(
        f"Fold {stratified_fold_number} "
        "validation patients:"
    )

    print(
        stratified_validation_patients_by_fold[
            stratified_fold_number
        ]
    )


Fold estratificado 1:
- Training patients: 23
- Validation patients: 3
- Shared patients: 0

Fold estratificado 2:
- Training patients: 21
- Validation patients: 5
- Shared patients: 0

Fold estratificado 3:
- Training patients: 20
- Validation patients: 6
- Shared patients: 0

Fold estratificado 4:
- Training patients: 21
- Validation patients: 5
- Shared patients: 0

Fold estratificado 5:
- Training patients: 19
- Validation patients: 7
- Shared patients: 0



,fold,train_signals,val_signals,train_patients,val_patients,train_positive,train_negative,val_positive,val_negative,train_positive_percentage,val_positive_percentage
0,1,7528,1831,23,3,1376,6152,364,1467,18.28,19.88
1,2,7407,1952,21,5,1240,6167,500,1452,16.74,25.61
2,3,7403,1956,20,6,1418,5985,322,1634,19.15,16.46
3,4,7473,1886,21,5,1494,5979,246,1640,19.99,13.04
4,5,7625,1734,19,7,1432,6193,308,1426,18.78,17.76


,fold,train_positive_patients,val_positive_patients
0,1,11,1
1,2,11,1
2,3,7,5
3,4,9,3
4,5,10,2


,fold,train_classes,val_classes,train_has_both_classes,val_has_both_classes
0,1,"[0, 1]","[0, 1]",True,True
1,2,"[0, 1]","[0, 1]",True,True
2,3,"[0, 1]","[0, 1]",True,True
3,4,"[0, 1]","[0, 1]",True,True
4,5,"[0, 1]","[0, 1]",True,True


Fold 1 validation patients:
['BH0002', 'BH0018', 'BH0103']
Fold 2 validation patients:
['BH0003', 'BH0014', 'BH0020', 'BH0035', 'BH0044']
Fold 3 validation patients:
['BH0007', 'BH0022', 'BH0023', 'BH0025', 'BH0028', 'BH0030']
Fold 4 validation patients:
['BH0004', 'BH0019', 'BH0027', 'BH0033', 'BH0101']
Fold 5 validation patients:
['BH0005', 'BH0006', 'BH0015', 'BH0016', 'BH0031', 'BH0032', 'BH0036']


#### 4.5.3.3 Final StratifiedGroupKFold summary

A compact summary of the five folds and the completed checks is displayed.

No individual fold is selected: all five folds are retained as one cross-validation strategy.

In [21]:
stratified_final_summary = (
    stratified_fold_summary_df[
        [
            "fold",
            "train_patients",
            "val_patients",
            "train_signals",
            "val_signals",
            "train_positive_percentage",
            "val_positive_percentage",
            "train_positive_patients",
            "val_positive_patients",
        ]
    ]
)

display(
    stratified_final_summary
)

stratified_no_patient_leakage = True

for stratified_fold_data in stratified_cv_folds:
    stratified_train_indices = (
        stratified_fold_data["train_indices"]
    )

    stratified_val_indices = (
        stratified_fold_data["val_indices"]
    )

    stratified_train_patients = set(
        groups_dev[
            stratified_train_indices
        ]
    )

    stratified_val_patients = set(
        groups_dev[
            stratified_val_indices
        ]
    )

    if not stratified_train_patients.isdisjoint(
        stratified_val_patients
    ):
        stratified_no_patient_leakage = False

print("Checks:")

print(
    "- Patient leakage: "
    f"{'NO' if stratified_no_patient_leakage else 'YES'}"
)

print(
    "- Both classes present in every training fold: "
    f"{'YES' if stratified_train_classes_all_present else 'NO'}"
)

print(
    "- Both classes present in every validation fold: "
    f"{'YES' if stratified_val_classes_all_present else 'NO'}"
)


print(
    "\nNo individual fold has been selected."
)

print(
    "The test set was not used."
)

,fold,train_patients,val_patients,train_signals,val_signals,train_positive_percentage,val_positive_percentage,train_positive_patients,val_positive_patients
0,1,23,3,7528,1831,18.28,19.88,11,1
1,2,21,5,7407,1952,16.74,25.61,11,1
2,3,20,6,7403,1956,19.15,16.46,7,5
3,4,21,5,7473,1886,19.99,13.04,9,3
4,5,19,7,7625,1734,18.78,17.76,10,2


Checks:
- Patient leakage: NO
- Both classes present in every training fold: YES
- Both classes present in every validation fold: YES

No individual fold has been selected.
The test set was not used.


`StratifiedGroupKFold` provides a modest improvement in the proportion of positive signals:

- `GroupKFold`: validation between 13.21% and 26.61%.
- `StratifiedGroupKFold`: validation between 13.04% and 25.61%.
- It also keeps training proportions relatively stable, between 16.74% and 19.99%.

It reasonably balances the total number of signals (`val_signals`),
but it does not reliably balance the number of positive patients (`val_positive_patients`).

Nevertheless, the improvement is limited.

### 4.5.4 Patient-level stratification

First, a one-row-per-patient table is constructed for the development set.

The `has_positive` variable indicates whether a patient contributes at least one positive signal. It is used as the patient-level stratification criterion.

In [22]:
# Obtain the unique patients in the development set.
patient_level_development_ids = np.unique(
    groups_dev
)

# Create a temporary table with the patient and label
# associated with each signal.
patient_level_labels_df = pd.DataFrame(
    {
        "patient_id": groups_dev,
        "label": y_dev,
    }
)

# Compute the total number of signals per patient.
patient_level_df = (
    patient_level_labels_df
    .groupby("patient_id")
    .agg(
        total_signals=("label", "size"),
        positive_signals=("label", "sum"),
    )
)

# Compute the number of negative signals.
patient_level_df["negative_signals"] = (
    patient_level_df["total_signals"]
    - patient_level_df["positive_signals"]
)

# Indicate whether the patient has any positive signal.
patient_level_df["has_positive"] = (
    patient_level_df["positive_signals"] > 0
).astype(int)

# Ordenamos la tabla por identificador de paciente.
patient_level_df = (
    patient_level_df
    .reset_index()
    .sort_values("patient_id")
    .reset_index(drop=True)
)

print(
    f"Development patients: "
    f"{len(patient_level_df)}"
)

print(
    "Patients with at least one positive signal:",
    (
        patient_level_df["has_positive"] == 1
    ).sum(),
)

print(
    "Patients with no positive signals:",
    (
        patient_level_df["has_positive"] == 0
    ).sum(),
)

display(
    patient_level_df
)

Development patients: 26
Patients with at least one positive signal: 12
Patients with no positive signals: 14


,patient_id,total_signals,positive_signals,negative_signals,has_positive
0,BH0002,1638,364,1274,1
1,BH0003,612,0,612,0
2,BH0004,378,126,252,1
3,BH0005,246,246,0,1
4,BH0006,252,0,252,0
5,BH0007,180,57,123,1
6,BH0014,252,0,252,0
7,BH0015,318,0,318,0
8,BH0016,72,0,72,0
9,BH0018,126,0,126,0


#### 4.5.4.1 Creating patient-level StratifiedKFold

`StratifiedKFold` is applied to a table with one row per patient, rather than to individual signals.

Therefore, stratification is based on whether each patient contributes at least one positive signal.

In [23]:
from sklearn.model_selection import StratifiedKFold

patient_ids = patient_level_df["patient_id"].to_numpy()
patient_labels = patient_level_df["has_positive"].to_numpy()

patient_stratified_kfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

patient_stratified_folds = []

for fold, (train_patient_idx, val_patient_idx) in enumerate(
    patient_stratified_kfold.split(patient_ids, patient_labels),
    start=1,
):
    train_patient_ids = patient_ids[train_patient_idx]
    val_patient_ids = patient_ids[val_patient_idx]

    patient_stratified_folds.append(
        {
            "fold": fold,
            "train_indices": np.flatnonzero(
                np.isin(groups_dev, train_patient_ids)
            ),
            "val_indices": np.flatnonzero(
                np.isin(groups_dev, val_patient_ids)
            ),
            "train_patient_ids": train_patient_ids,
            "val_patient_ids": val_patient_ids,
        }
    )

print(f"Folds generated: {len(patient_stratified_folds)}")

Folds generated: 5


#### 4.5.4.2 Checking for patient leakage

This section verifies that training and validation patients are completely independent in every fold.

In [24]:
for patient_stratified_fold_data in (
    patient_stratified_folds
):
    patient_stratified_fold_number = (
        patient_stratified_fold_data["fold"]
    )

    patient_stratified_train_indices = (
        patient_stratified_fold_data[
            "train_indices"
        ]
    )

    patient_stratified_val_indices = (
        patient_stratified_fold_data[
            "val_indices"
        ]
    )

    patient_stratified_train_patient_ids = (
        patient_stratified_fold_data[
            "train_patient_ids"
        ]
    )

    patient_stratified_val_patient_ids = (
        patient_stratified_fold_data[
            "val_patient_ids"
        ]
    )

    # Check that no patients are shared.
    patient_stratified_shared_ids = set(
        patient_stratified_train_patient_ids
    ).intersection(
        set(patient_stratified_val_patient_ids)
    )

    assert set(
        patient_stratified_train_patient_ids
    ).isdisjoint(
        set(patient_stratified_val_patient_ids)
    )

    # Check that all signals are used.
    assert (
        len(patient_stratified_train_indices)
        + len(patient_stratified_val_indices)
        == len(X_dev)
    )

    # Check that no indices are repeated.
    assert set(
        patient_stratified_train_indices
    ).isdisjoint(
        set(patient_stratified_val_indices)
    )
    # Comprobamos la cobertura completa de X_dev
        # en el fold actual.
    current_fold_indices = np.concatenate(
        [
            patient_stratified_train_indices,
            patient_stratified_val_indices,
        ]
    )

    assert len(current_fold_indices) == len(X_dev)

    assert len(
        np.unique(current_fold_indices)
    ) == len(X_dev)

    assert set(current_fold_indices) == set(
        range(len(X_dev))
    )  

    print(
        f"Fold {patient_stratified_fold_number}:"
    )

    print(
        "- Training patients:",
        len(patient_stratified_train_patient_ids),
    )

    print(
        "- Validation patients:",
        len(patient_stratified_val_patient_ids),
    )

    print(
        "- Shared patients:",
        len(patient_stratified_shared_ids),
    )


assert len(patient_level_df) == len(np.unique(groups_dev))
assert len(patient_level_df["has_positive"].unique()) == 2
assert (
    patient_level_df["has_positive"].value_counts().min()
    >= patient_stratified_kfold.n_splits
)

Fold 1:
- Training patients: 20
- Validation patients: 6
- Shared patients: 0
Fold 2:
- Training patients: 21
- Validation patients: 5
- Shared patients: 0
Fold 3:
- Training patients: 21
- Validation patients: 5
- Shared patients: 0
Fold 4:
- Training patients: 21
- Validation patients: 5
- Shared patients: 0
Fold 5:
- Training patients: 21
- Validation patients: 5
- Shared patients: 0


#### 4.5.4.3 Signal and class summary by fold

The number of patients, signals, positive signals, negative signals, and positive-signal percentage is computed for training and validation.

In [25]:
patient_stratified_fold_summary_records = []
#
for patient_stratified_fold_data in (
    patient_stratified_folds
):
    patient_stratified_fold_number = (
        patient_stratified_fold_data["fold"]
    )

    patient_stratified_train_indices = (
        patient_stratified_fold_data[
            "train_indices"
        ]
    )

    patient_stratified_val_indices = (
        patient_stratified_fold_data[
            "val_indices"
        ]
    )

    patient_stratified_train_labels = y_dev[
        patient_stratified_train_indices
    ]

    patient_stratified_val_labels = y_dev[
        patient_stratified_val_indices
    ]

    patient_stratified_train_positive = (
        patient_stratified_train_labels == 1
    ).sum()

    patient_stratified_train_negative = (
        patient_stratified_train_labels == 0
    ).sum()

    patient_stratified_val_positive = (
        patient_stratified_val_labels == 1
    ).sum()

    patient_stratified_val_negative = (
        patient_stratified_val_labels == 0
    ).sum()

    patient_stratified_fold_summary_records.append(
        {
            "fold": patient_stratified_fold_number,
            "train_patients": len(
                patient_stratified_fold_data[
                    "train_patient_ids"
                ]
            ),
            "val_patients": len(
                patient_stratified_fold_data[
                    "val_patient_ids"
                ]
            ),
            "train_signals": len(
                patient_stratified_train_indices
            ),
            "val_signals": len(
                patient_stratified_val_indices
            ),
            "train_positive": (
                patient_stratified_train_positive
            ),
            "train_negative": (
                patient_stratified_train_negative
            ),
            "val_positive": (
                patient_stratified_val_positive
            ),
            "val_negative": (
                patient_stratified_val_negative
            ),
            "train_positive_percentage": (
                100
                * patient_stratified_train_positive
                / len(
                    patient_stratified_train_indices
                )
            ),
            "val_positive_percentage": (
                100
                * patient_stratified_val_positive
                / len(
                    patient_stratified_val_indices
                )
            ),
        }
    )

patient_stratified_fold_summary_df = pd.DataFrame(
    patient_stratified_fold_summary_records
)

patient_stratified_fold_summary_df[
    [
        "train_positive_percentage",
        "val_positive_percentage",
    ]
] = patient_stratified_fold_summary_df[
    [
        "train_positive_percentage",
        "val_positive_percentage",
    ]
].round(2)

display(
    patient_stratified_fold_summary_df
)

,fold,train_patients,val_patients,train_signals,val_signals,train_positive,train_negative,val_positive,val_negative,train_positive_percentage,val_positive_percentage
0,1,20,6,7936,1423,1352,6584,388,1035,17.04,27.27
1,2,21,5,7042,2317,1120,5922,620,1697,15.90,26.76
2,3,21,5,8212,1147,1560,6652,180,967,19.00,15.69
3,4,21,5,7991,1368,1557,6434,183,1185,19.48,13.38
4,5,21,5,6255,3104,1371,4884,369,2735,21.92,11.89


#### 4.5.4.4 Patients contributing positive signals

The number of patients contributing at least one positive signal in training and validation is counted for each fold.

In [26]:
patient_stratified_train_positive_counts = []
patient_stratified_val_positive_counts = []

for patient_stratified_fold_data in (
    patient_stratified_folds
):
    patient_stratified_train_indices = (
        patient_stratified_fold_data[
            "train_indices"
        ]
    )

    patient_stratified_val_indices = (
        patient_stratified_fold_data[
            "val_indices"
        ]
    )

    patient_stratified_train_labels = y_dev[
        patient_stratified_train_indices
    ]

    patient_stratified_val_labels = y_dev[
        patient_stratified_val_indices
    ]

    patient_stratified_train_groups = groups_dev[
        patient_stratified_train_indices
    ]

    patient_stratified_val_groups = groups_dev[
        patient_stratified_val_indices
    ]

    # Positive training patients.
    patient_stratified_train_positive_patients = (
        np.unique(
            patient_stratified_train_groups[
                patient_stratified_train_labels == 1
            ]
        )
    )

    # Positive validation patients.
    patient_stratified_val_positive_patients = (
        np.unique(
            patient_stratified_val_groups[
                patient_stratified_val_labels == 1
            ]
        )
    )

    patient_stratified_train_positive_counts.append(
        len(
            patient_stratified_train_positive_patients
        )
    )

    patient_stratified_val_positive_counts.append(
        len(
            patient_stratified_val_positive_patients
        )
    )

patient_stratified_fold_summary_df[
    "train_positive_patients"
] = patient_stratified_train_positive_counts

patient_stratified_fold_summary_df[
    "val_positive_patients"
] = patient_stratified_val_positive_counts

display(
    patient_stratified_fold_summary_df[
        [
            "fold",
            "train_positive_patients",
            "val_positive_patients",
        ]
    ]
)

,fold,train_positive_patients,val_positive_patients
0,1,9,3
1,2,9,3
2,3,10,2
3,4,10,2
4,5,10,2


#### 4.5.4.5 Checking the presence of both classes

The classes present in training and validation are checked for each fold.

The patient assignments are not modified if a subset does not contain both classes.

In [27]:
patient_stratified_class_presence_records = []

patient_stratified_train_classes_all_present = True
patient_stratified_val_classes_all_present = True

for patient_stratified_fold_data in (
    patient_stratified_folds
):
    patient_stratified_fold_number = (
        patient_stratified_fold_data["fold"]
    )

    patient_stratified_train_indices = (
        patient_stratified_fold_data[
            "train_indices"
        ]
    )

    patient_stratified_val_indices = (
        patient_stratified_fold_data[
            "val_indices"
        ]
    )

    patient_stratified_train_classes = np.unique(
        y_dev[
            patient_stratified_train_indices
        ]
    )

    patient_stratified_val_classes = np.unique(
        y_dev[
            patient_stratified_val_indices
        ]
    )

    patient_stratified_train_has_both_classes = (
        set(
            patient_stratified_train_classes
        )
        == {0, 1}
    )

    patient_stratified_val_has_both_classes = (
        set(
            patient_stratified_val_classes
        )
        == {0, 1}
    )

    if not patient_stratified_train_has_both_classes:
        patient_stratified_train_classes_all_present = (
            False
        )

    if not patient_stratified_val_has_both_classes:
        patient_stratified_val_classes_all_present = (
            False
        )

    patient_stratified_class_presence_records.append(
        {
            "fold": patient_stratified_fold_number,
            "train_classes": (
                patient_stratified_train_classes
            ),
            "val_classes": (
                patient_stratified_val_classes
            ),
            "train_has_both_classes": (
                patient_stratified_train_has_both_classes
            ),
            "val_has_both_classes": (
                patient_stratified_val_has_both_classes
            ),
        }
    )

patient_stratified_class_presence_df = pd.DataFrame(
    patient_stratified_class_presence_records
)

display(
    patient_stratified_class_presence_df
)

if not patient_stratified_train_classes_all_present:
    print(
        "Warning: a training fold "
        "does not contain both classes."
    )

if not patient_stratified_val_classes_all_present:
    print(
        "Warning: a validation fold "
        "does not contain both classes."
    )

,fold,train_classes,val_classes,train_has_both_classes,val_has_both_classes
0,1,"[0, 1]","[0, 1]",True,True
1,2,"[0, 1]","[0, 1]",True,True
2,3,"[0, 1]","[0, 1]",True,True
3,4,"[0, 1]","[0, 1]",True,True
4,5,"[0, 1]","[0, 1]",True,True


#### 4.5.4.6 Validation patients by fold

The patients assigned to validation are displayed for each fold.

The number of validation patients contributing positive signals is also displayed.

In [28]:
patient_stratified_validation_patients_by_fold = {}

for patient_stratified_fold_data in (
    patient_stratified_folds
):
    patient_stratified_fold_number = (
        patient_stratified_fold_data["fold"]
    )

    patient_stratified_val_patient_ids = (
        patient_stratified_fold_data[
            "val_patient_ids"
        ]
    )

    patient_stratified_val_positive_patients = (
        patient_stratified_fold_summary_df
        .loc[
            patient_stratified_fold_summary_df[
                "fold"
            ]
            == patient_stratified_fold_number,
            "val_positive_patients",
        ]
        .iloc[0]
    )

    patient_stratified_validation_patients_by_fold[
        patient_stratified_fold_number
    ] = sorted(
        patient_stratified_val_patient_ids
    )

    print(
        f"Fold {patient_stratified_fold_number} "
        "validation patients:"
    )

    print(
        patient_stratified_validation_patients_by_fold[
            patient_stratified_fold_number
        ]
    )

    print(
        "- Validation patients with positive signals:",
        patient_stratified_val_positive_patients,
    )



Fold 1 validation patients:
['BH0005', 'BH0014', 'BH0022', 'BH0023', 'BH0031', 'BH0044']
- Validation patients with positive signals: 3
Fold 2 validation patients:
['BH0019', 'BH0020', 'BH0027', 'BH0032', 'BH0035']
- Validation patients with positive signals: 3
Fold 3 validation patients:
['BH0015', 'BH0030', 'BH0033', 'BH0036', 'BH0103']
- Validation patients with positive signals: 2
Fold 4 validation patients:
['BH0003', 'BH0004', 'BH0007', 'BH0016', 'BH0018']
- Validation patients with positive signals: 2
Fold 5 validation patients:
['BH0002', 'BH0006', 'BH0025', 'BH0028', 'BH0101']
- Validation patients with positive signals: 2


#### 4.5.4.7 Checking complete patient coverage

This section verifies that every development patient appears exactly once as a validation patient across the five folds.

In [29]:
patient_stratified_all_validation_patients = []

for patient_stratified_fold_data in (
    patient_stratified_folds
):
    patient_stratified_all_validation_patients.extend(
        patient_stratified_fold_data[
            "val_patient_ids"
        ]
    )

patient_stratified_unique_validation_patients = set(
    patient_stratified_all_validation_patients
)

patient_stratified_validation_patient_counts = (
    pd.Series(
        patient_stratified_all_validation_patients
    )
    .value_counts()
)

patient_stratified_repeated_validation_patients = (
    patient_stratified_validation_patient_counts[
        patient_stratified_validation_patient_counts > 1
    ]
)

patient_stratified_all_development_patients = set(
    patient_level_development_ids
)

patient_stratified_all_patients_are_covered = (
    patient_stratified_unique_validation_patients
    == patient_stratified_all_development_patients
)

patient_stratified_each_patient_appears_once = (
    len(
        patient_stratified_repeated_validation_patients
    )
    == 0
)

assert len(patient_stratified_all_validation_patients) == len(
    patient_stratified_unique_validation_patients
)

assert (
    patient_stratified_unique_validation_patients
    == patient_stratified_all_development_patients
)

print(
    f"Development patients: "
    f"{len(patient_stratified_all_development_patients)}"
)

print(
    f"Patients appearing in validation: "
    f"{len(patient_stratified_unique_validation_patients)}"
)

print(
    f"Repeated patients in validation: "
    f"{len(patient_stratified_repeated_validation_patients)}"
)

print(
    "All patients appear exactly "
    "once in validation: "
    f"{'YES' if patient_stratified_all_patients_are_covered and patient_stratified_each_patient_appears_once else 'NO'}"
)

Development patients: 26
Patients appearing in validation: 26
Repeated patients in validation: 0
All patients appear exactly once in validation: YES


#### 4.5.4.8 Final patient-level stratification summary

A summary of the five folds generated by patient-level `StratifiedKFold` is displayed.

No individual fold is selected. These folds remain available for comparison with the grouped strategies.

In [30]:
patient_stratified_final_summary = (
    patient_stratified_fold_summary_df[
        [
            "fold",
            "train_patients",
            "val_patients",
            "train_signals",
            "val_signals",
            "train_positive_percentage",
            "val_positive_percentage",
            "train_positive_patients",
            "val_positive_patients",
        ]
    ]    
)
display(
    patient_stratified_final_summary
)



patient_stratified_no_patient_leakage = True

for patient_stratified_fold_data in (
    patient_stratified_folds
):
    patient_stratified_train_patient_ids = set(
        patient_stratified_fold_data[
            "train_patient_ids"
        ]
    )

    patient_stratified_val_patient_ids = set(
        patient_stratified_fold_data[
            "val_patient_ids"
        ]
    )

    if not patient_stratified_train_patient_ids.isdisjoint(
        patient_stratified_val_patient_ids
    ):
        patient_stratified_no_patient_leakage = False

print("Checks:")

print(
    "- Patient leakage: "
    f"{'NO' if patient_stratified_no_patient_leakage else 'YES'}"
)

print(
    "- All patients appear exactly "
    f"una vez en validation: "
    f"{'YES' if patient_stratified_all_patients_are_covered and patient_stratified_each_patient_appears_once else 'NO'}"
)

print(
    "- Both classes present in every training fold: "
    f"{'YES' if patient_stratified_train_classes_all_present else 'NO'}"
)

print(
    "- Both classes present in every validation fold: "
    f"{'YES' if patient_stratified_val_classes_all_present else 'NO'}"
)

print(
    "\nNo individual fold has been selected."
)

print(
    "GroupKFold and "
    "StratifiedGroupKFold variables were not overwritten."
)

,fold,train_patients,val_patients,train_signals,val_signals,train_positive_percentage,val_positive_percentage,train_positive_patients,val_positive_patients
0,1,20,6,7936,1423,17.04,27.27,9,3
1,2,21,5,7042,2317,15.90,26.76,9,3
2,3,21,5,8212,1147,19.00,15.69,10,2
3,4,21,5,7991,1368,19.48,13.38,10,2
4,5,21,5,6255,3104,21.92,11.89,10,2


Checks:
- Patient leakage: NO
- All patients appear exactly una vez en validation: YES
- Both classes present in every training fold: YES
- Both classes present in every validation fold: YES

No individual fold has been selected.
GroupKFold and StratifiedGroupKFold variables were not overwritten.


Patient-level stratification balances the number of positive patients more directly.

The main difference is:
- `StratifiedGroupKFold` at signal level: better balance between positive and negative signals, although positive patients are not perfectly balanced.
- Patient-level `StratifiedKFold`: better balance in the number of positive patients, but weaker balance in the number and proportion of positive signals.

## 4.6 Saving and verifying the final split files
This section saves the development/test split, the `StratifiedGroupKFold` fold indices used by the downstream pipeline, and their metadata.
The patient-level `StratifiedKFold` results remain available in memory for comparison.

### 4.6.1. Defining the output directory

An output directory is created for the split files and their metadata.

The path is relative, and the signals and original files are not modified.

In [31]:
# Relative path where the split files will be saved.
# Use Path("data") / "splits" when running the notebook
# from the TurraFM project root.
splits_path = (
    Path("data")
    / "splits"
)

# Create the directory if it does not exist yet.
splits_path.mkdir(
    parents=True,
    exist_ok=True,
)

print(
    f"Output directory: "
    f"{splits_path.resolve()}"
)

Output directory: /Users/ainhoa.ns/Desktop/TurraFM/-Reconocimiento-de-patrones-en-se-ales-ECG-con-ML/notebooks/data/splits


### 4.6.2. Preparing the fold indices

The indices of the five `StratifiedGroupKFold` folds are extracted for downstream use.

The folds are not recalculated. The patient-level folds remain separate and are not the folds saved by the following cells.

In [32]:
# Extract the fold indices without recalculating them.
assert len(stratified_cv_folds) == 5

# Store the general split arrays.
split_arrays = {
    "development_indices": np.asarray(
        development_indices,
        dtype=np.int64,
    ),
    "test_indices": np.asarray(
        test_indices,
        dtype=np.int64,
    ),
}
# Store the training and validation indices
# de cada fold de StratifiedGroupKFold.
for fold_data in stratified_cv_folds:
    fold_number = fold_data["fold"]

    split_arrays[
        f"fold_{fold_number}_train_indices"
    ] = np.asarray(
        fold_data["train_indices"],
        dtype=np.int64,
    )

    split_arrays[
        f"fold_{fold_number}_val_indices"
    ] = np.asarray(
        fold_data["val_indices"],
        dtype=np.int64,
    )

print(
    "The five fold indices have been extracted."
)

print(
    "All are NumPy arrays:",
    all(
        isinstance(
            stratified_cv_folds[fold_number][
                "train_indices"
            ],
            np.ndarray,
        )
        and isinstance(
            stratified_cv_folds[fold_number][
                "val_indices"
            ],
            np.ndarray,
        )
        for fold_number in range(5)
    ),
)

The five fold indices have been extracted.
All are NumPy arrays: True


### 4.6.3. Saving the split arrays

`dataset_splits.npz` stores the development and test indices, together with the training and validation indices for each fold.

The `development_indices` and `test_indices` refer to the complete dataset. The five fold indices are relative exclusively to `X_dev`.

In [33]:
# Convert the final partitions to NumPy arrays.
saved_development_indices = np.asarray(
    development_indices,
    dtype=np.int64,
)

saved_test_indices = np.asarray(
    test_indices,
    dtype=np.int64,
)


dataset_splits_path = (
    splits_path
    / "dataset_splits.npz"
)

# Save the partitions without modifying X or the original files.
np.savez_compressed(
    dataset_splits_path,
    **split_arrays,
)
loaded_splits = np.load(
    dataset_splits_path
)
print(
    f"Splits saved to: "
    f"{dataset_splits_path}"
)

Splits saved to: data/splits/dataset_splits.npz


### 4.6.4. Saving split metadata

The random seeds, number of folds, and development, test, and validation patient identifiers are saved.

This metadata documents the strategy and supports its reconstruction later.

In [34]:
split_metadata_path = (
    splits_path
    / "split_metadata.npz"
)

# Load indices relative to the complete dataset.
loaded_development_indices = loaded_splits[
    "development_indices"
]

loaded_test_indices = loaded_splits[
    "test_indices"
]

# Check that development and test do not share indices.
assert set(
    loaded_development_indices
).isdisjoint(
    set(loaded_test_indices)
)

# Check that together they cover all dataset signals.
all_dataset_indices = np.concatenate(
    [
        loaded_development_indices,
        loaded_test_indices,
    ]
)

assert len(all_dataset_indices) == len(X)
assert len(np.unique(all_dataset_indices)) == len(X)
assert set(all_dataset_indices) == set(
    np.arange(len(X))
)

print(
    "Development and test cover the complete dataset "
    "without overlap."
)

# Check the coverage of each fold over X_dev.
for fold_number in range(1, 6):
    loaded_train_indices = loaded_splits[
        f"fold_{fold_number}_train_indices"
    ]

    loaded_val_indices = loaded_splits[
        f"fold_{fold_number}_val_indices"
    ]

    assert set(
        loaded_train_indices
    ).isdisjoint(
        set(loaded_val_indices)
    )

    fold_indices = np.concatenate(
        [
            loaded_train_indices,
            loaded_val_indices,
        ]
    )

    assert len(fold_indices) == len(X_dev)
    assert len(np.unique(fold_indices)) == len(X_dev)
    assert set(fold_indices) == set(
        np.arange(len(X_dev))
    )

    print(
        f"Fold {fold_number}: X_dev coverage "
        "is correct."
    )

Development and test cover the complete dataset without overlap.
Fold 1: X_dev coverage is correct.
Fold 2: X_dev coverage is correct.
Fold 3: X_dev coverage is correct.
Fold 4: X_dev coverage is correct.
Fold 5: X_dev coverage is correct.


### 4.6.5. Verifying the saved split files

The split file is reloaded and its saved indices are compared with the final in-memory indices.

In [35]:
# Reload the split file.
loaded_splits.close()

loaded_splits = np.load(
    dataset_splits_path
) 

print(
    "Saved file keys:"
)

print(
    loaded_splits.files
)

# Check the development/test partitions.
assert np.array_equal(
    loaded_splits["development_indices"],
    development_indices,
)

assert np.array_equal(
    loaded_splits["test_indices"],
    test_indices,
)


print(
    "All partitions were loaded "
    "successfully."
)

print(
    "The saved partitions match "
    "the original partitions."
)

Saved file keys:
['development_indices', 'test_indices', 'fold_1_train_indices', 'fold_1_val_indices', 'fold_2_train_indices', 'fold_2_val_indices', 'fold_3_train_indices', 'fold_3_val_indices', 'fold_4_train_indices', 'fold_4_val_indices', 'fold_5_train_indices', 'fold_5_val_indices']
All partitions were loaded successfully.
The saved partitions match the original partitions.


### 4.6.6. Checking the saved patient metadata

The patient identifiers saved in `split_metadata.npz` are compared with the patients used by `stratified_cv_folds`.

In [36]:
# Load the split metadata.
split_metadata = np.load(
    split_metadata_path
)
print(Path.cwd())
print(splits_path.resolve())

print(
    "Available keys in split_metadata.npz:"
)

print(
    split_metadata.files
)

# Display the development and test patients.
saved_test_patient_ids = (
    split_metadata[
            "test_patient_ids"
        ]
)

saved_development_patient_ids = (
    split_metadata[
        "development_patient_ids"
    ]
)

split_metadata = {
    "test_seed": np.array(
        43,
        dtype=np.int64,
    ),
    "cv_seed": np.array(
        43,
        dtype=np.int64,
    ),
    "number_of_folds": np.array(
        5,
        dtype=np.int64,
    ),
    "cv_strategy": np.array(
        "StratifiedGroupKFold"
    ),
    "test_patient_ids": np.unique(
        groups_test
    ).astype(str),
    "development_patient_ids": np.unique(
        groups_dev
    ).astype(str),
}

print("\nPatients saved in test:")
print(
    sorted(saved_test_patient_ids.tolist())
)

print("\nPatients saved in development:")
print(
    sorted(saved_development_patient_ids.tolist())
)

for fold_data in stratified_cv_folds:
    fold_number = fold_data["fold"]

    validation_indices = (
        fold_data["val_indices"]
    )

    validation_patient_ids = np.unique(
        groups_dev[
            validation_indices
        ]
    ).astype(str)

    split_metadata[
        f"fold_{fold_number}_val_patient_ids"
    ] = validation_patient_ids

np.savez_compressed(
    split_metadata_path,
    **split_metadata,
)

print(
    f"Metadata saved to:\n"
    f"{split_metadata_path}"
)

/Users/ainhoa.ns/Desktop/TurraFM/-Reconocimiento-de-patrones-en-se-ales-ECG-con-ML/notebooks
/Users/ainhoa.ns/Desktop/TurraFM/-Reconocimiento-de-patrones-en-se-ales-ECG-con-ML/notebooks/data/splits
Available keys in split_metadata.npz:
['test_seed', 'cv_seed', 'number_of_folds', 'cv_strategy', 'test_patient_ids', 'development_patient_ids', 'fold_1_val_patient_ids', 'fold_2_val_patient_ids', 'fold_3_val_patient_ids', 'fold_4_val_patient_ids', 'fold_5_val_patient_ids']

Patients saved in test:
['BH0008', 'BH0009', 'BH0010', 'BH0012', 'BH0017', 'BH0026', 'BH0029']

Patients saved in development:
['BH0002', 'BH0003', 'BH0004', 'BH0005', 'BH0006', 'BH0007', 'BH0014', 'BH0015', 'BH0016', 'BH0018', 'BH0019', 'BH0020', 'BH0022', 'BH0023', 'BH0025', 'BH0027', 'BH0028', 'BH0030', 'BH0031', 'BH0032', 'BH0033', 'BH0035', 'BH0036', 'BH0044', 'BH0101', 'BH0103']
Metadata saved to:
data/splits/split_metadata.npz


In [2]:
loaded_metadata = np.load(
    split_metadata_path
)

print(
    loaded_metadata.files
)
# Compare the patients saved for each fold
# with the patients computed from stratified_cv_folds.

for fold_number in range(1, 6):

    # Patients saved in the .npz file.
    saved_validation_patient_ids = np.asarray(
        loaded_metadata[
            f"fold_{fold_number}_val_patient_ids"
        ],
        dtype=str,
    )

    # Validation indices for the current fold.
    current_validation_indices = (
        stratified_cv_folds[
            fold_number - 1
        ]["val_indices"]
    )

    # Obtain patients from the indices.
    current_validation_patient_ids = np.unique(
        groups_dev[
            current_validation_indices
        ]
    ).astype(str)

    same_validation_patients = (
        set(saved_validation_patient_ids)
        == set(current_validation_patient_ids)
    )

    print(
        f"Fold {fold_number}: "
        f"{same_validation_patients}"
    )

    print(
        "Saved patients:"
    )

    print(
        sorted(
            saved_validation_patient_ids.tolist()
        )
    )

    print(
        "Current patients:"
    )

    print(
        sorted(
            current_validation_patient_ids.tolist()
        )
    )


NameError: name 'np' is not defined

### 4.6.7. Final summary

The notebook creates a patient-level development/test split, compares three cross-validation strategies, and saves the `StratifiedGroupKFold` folds used by the downstream pipeline.

| Output | Description |
|---|---|
| `development_indices` | Signals assigned to the development subset. |
| `test_indices` | Signals assigned to the independent patient-level test subset. |
| `fold_*_train_indices` | Training indices relative to `X_dev`. |
| `fold_*_val_indices` | Validation indices relative to `X_dev`. |
| `fold_*_val_patient_ids` | Validation patients stored in the metadata file. |

The saved partitions are checked for complete coverage, absence of overlap, patient-level independence, and consistency after reloading. No signals or PyTorch tensors are saved.

In [38]:
print("Split files saved successfully.")
print()

print(
    f"Split file: "
    f"{dataset_splits_path}"
)

print(
    f"Metadata file: "
    f"{split_metadata_path}"
)

print()
print("Complete dataset:")
print("- Development indices saved")
print("- Test indices saved")

print()
print("Cross-validation:")
print("- Five folds saved")
print("- Training/validation indices for each fold")

print()
print("Metadata:")
print("- Test seed: 43")
print("- Cross-validation seed: 43")
print("- Test patients saved")

print()
print("Integrity verification: successful")

Split files saved successfully.

Split file: data/splits/dataset_splits.npz
Metadata file: data/splits/split_metadata.npz

Complete dataset:
- Development indices saved
- Test indices saved

Cross-validation:
- Five folds saved
- Training/validation indices for each fold

Metadata:
- Test seed: 43
- Cross-validation seed: 43
- Test patients saved

Integrity verification: successful


# Variable reference
| Variable | Type/format | Expected content |

| `nonormalized_data_path` | `pathlib.Path` | Path to `data/nonormalized_signals_clean.npz`. |
| `nonormalized_data` | `numpy.lib.npyio.NpzFile` | Open `.npz` file containing `signals`, `patient_id`, `lead`, `label`, and `mat_path`. |
| `X` | `numpy.ndarray` | Non-normalized ECG signals with shape `(10577, 252)`. |
| `y_original` | `numpy.ndarray` | Original labels with values `1` or `99`. |
| `Y` | `numpy.ndarray` | Binary labels: `1` for Brugada and `0` for non-Brugada. |
| `patient_ids` | `numpy.ndarray` | Patient identifier associated with each signal. |
| `leads` | `numpy.ndarray` | Lead associated with each signal, such as `V1` or `V2`. |
| `dataset_index` | `pandas.DataFrame` | One row per signal with patient, lead, label, and source path. |
| `groups` | `numpy.ndarray` | Copy of `patient_ids`, used to split by patient and prevent leakage. |
| `unique_patient_ids` | `numpy.ndarray` | Array of unique patients. |
| `development_ids`, `test_ids` | `numpy.ndarray` | Patient identifiers assigned to development and test. |
| `test_mask`, `development_mask` | `numpy.ndarray` | Boolean masks identifying signals assigned to each subset. |


| `development_indices`, `test_indices` | `numpy.ndarray` | Signal indices assigned to development and test. |
| `X_dev`, `y_dev` | `numpy.ndarray` | Development signals and binary labels. |
| `groups_dev`, `leads_dev` | `numpy.ndarray` | Patient and lead identifiers for development signals. |
| `X_test`, `y_test` | `numpy.ndarray` | Test signals and binary labels. |
| `groups_test`, `leads_test` | `numpy.ndarray` | Patient and lead identifiers for test signals. |